# Sunete din text

Generăm sunete pornind de la o descriere, cu AudioLDM 2, un model de difuzie latentă pentru audio. Urmărim drumul de la reprezentarea latentă la spectrograma mel și de la spectrogramă, prin vocoder, la sunet.

Codul din acest notebook vine de pe pagina [Sunete din text](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/sunete-din-text/) a laboratorului [Imagini și sunete cu modele de difuzie](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro). Explicațiile fiecărui pas sunt pe pagină.

În Google Colab, codul rulează mult mai repede pe o placă grafică. Dacă nu este deja aleasă, se alege din meniul **Runtime**, opțiunea **Change runtime type**, o variantă cu GPU. Celulele se rulează în ordine, de sus în jos.

In [ ]:
%pip install -q -U diffusers transformers accelerate

## Primul sunet

Codul de mai jos generează 10 secunde de sunet pornind de la o descriere și le salvează într-un fișier WAV.

In [ ]:
import scipy.io.wavfile
import torch
from diffusers import AudioLDM2Pipeline

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32

# La prima rulare, fișierele modelului (aproximativ 4,5 GB) se descarcă de pe Hugging Face Hub.
pipe = AudioLDM2Pipeline.from_pretrained("cvssp/audioldm2", dtype=dtype).to(device)

prompt = "a dog barking in the distance while rain falls on a tin roof"
generator = torch.Generator("cpu").manual_seed(0)
audio = pipe(prompt, negative_prompt="low quality", audio_length_in_s=10.0, generator=generator).audios[0]

rate = pipe.vocoder.config.sampling_rate  # eșantioane pe secundă
print(f"{len(audio)} de eșantioane, adică {len(audio) / rate:.1f} secunde la {rate} Hz")
scipy.io.wavfile.write("sunet.wav", rate=rate, data=audio)

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Audio("sunet.wav"))

## De la reprezentarea latentă la sunet

Cu `output_type="latent"`, pipeline-ul se oprește după procesul invers și întoarce reprezentarea latentă curată, fără să o decodeze. Codul de mai jos face de mână ultimii doi pași, decodarea cu VAE-ul și vocoderul, și desenează spectrograma mel dintre ei.

In [ ]:
import matplotlib.pyplot as plt
import scipy.io.wavfile
import torch
from diffusers import AudioLDM2Pipeline

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
pipe = AudioLDM2Pipeline.from_pretrained("cvssp/audioldm2", dtype=dtype).to(device)

prompt = "a slow piano melody with soft rain in the background"
generator = torch.Generator("cpu").manual_seed(0)
latents = pipe(
    prompt, negative_prompt="low quality", audio_length_in_s=10.0, generator=generator, output_type="latent"
).audios
with torch.no_grad():
    mel = pipe.vae.decode(latents / pipe.vae.config.scaling_factor).sample  # spectrograma mel
    audio = pipe.mel_spectrogram_to_waveform(mel)[0].numpy()  # vocoderul HiFi-GAN
print("Reprezentarea latentă:", tuple(latents.shape))
print("Spectrograma mel:     ", tuple(mel.shape))
print("Forma de undă:        ", audio.shape)

plt.figure(figsize=(10, 3))
plt.imshow(mel[0, 0].T.float().cpu().numpy(), origin="lower", aspect="auto", cmap="magma")
plt.xlabel("cadre de câte 10 ms")
plt.ylabel("benzi mel")
plt.savefig("spectrograma.png", bbox_inches="tight")
scipy.io.wavfile.write("pian.wav", rate=pipe.vocoder.config.sampling_rate, data=audio)

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Audio("pian.wav"))